In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.load_data import get_validated_data
from src.edit_plot import add_bar_value

In [ ]:
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_colwidth", None)

In [ ]:
# Load the validated data
data = get_validated_data()
data.head(5)

## Univariate Analysis

In [ ]:
# General information
print(
    f"\n---Description--- \n{data[['release_date', 'price', 'recommendations']].describe()}"
)

In [ ]:
# Release date analysis
release_years_count = data["release_year"].value_counts()
release_dates = data["release_date"].value_counts()
mean, mode, median, max_date, min_date = (
    release_dates.mean(),
    release_dates.mode()[0],
    release_dates.median(),
    release_dates.max(),
    release_dates.min(),
)
popular_release_dates = release_dates[release_dates > mean].count()
releases_per_month = data["release_date"].dt.month.value_counts()
releases_per_year_month = (
    data.groupby(["release_year", data["release_date"].dt.month])
    .size()
    .reset_index(name="count")
)
release_date_analysis = f"""
Release Date Analysis
================================================
Count of days with releases:     {release_dates.count()}
Count of days with above-average releases: {popular_release_dates}
Mean:    {mean:.0f}
Mode:     {mode}
Median:  {median:.0f}
Max:   {max_date}
Min:   {min_date}
"""
print(release_date_analysis)

# Visual Release date analysis
sns.set_style("darkgrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("Release Date Analysis", fontsize=16)

years_plot = sns.barplot(
    x=release_years_count.index,
    y=release_years_count.values,
    ax=axes[0, 0],
    palette="Dark2",
)
axes[0, 0].set_title("Number of Games Released Each Year")
axes[0, 0].set(xlabel=None, ylabel="Releases")
add_bar_value(years_plot)

months_plot = sns.barplot(
    x=releases_per_month.index,
    y=releases_per_month.values,
    ax=axes[0, 1],
    palette="dark",
)
axes[0, 1].set_title("Number of Games Released in 2021-2025 Per Month")
axes[0, 1].set(xlabel=None, ylabel="Releases")
axes[0, 1].set_xticklabels(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
add_bar_value(months_plot)

sns.lineplot(
    data=releases_per_year_month,
    x="release_date",
    y="count",
    hue="release_year",
    ax=axes[1, 0],
    palette="Dark2",
)
axes[1, 0].set_title("Number of Games Released Over Time")
axes[1, 0].set(xlabel=None, ylabel="Releases")
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
axes[1, 0].legend(bbox_to_anchor=(0.02, 0.95), loc="upper left")
plt.tight_layout()

sns.boxplot(release_dates.values, ax=axes[1, 1], palette="Dark2")
axes[1, 1].set(xlabel=None, ylabel="Releases")
axes[1, 1].set_title("Distribution of Daily Releases")

plt.show()

In [ ]:
# Genres analysis
splitted_genres = data["genres"].str.split(";")
splitted_genres = splitted_genres[splitted_genres.str.len() > 1]
genres_combinations = splitted_genres.value_counts().reset_index(name="count")

most_popular_genres_combinations = genres_combinations.head(10)
most_popular_genres_combinations["genres"] = most_popular_genres_combinations["genres"].str.join(";")

genres_cobinations_count = (
    genres_combinations["count"].value_counts().reset_index(name="frequency")
)
genres_count = splitted_genres.explode().value_counts()

gencomb_mean, gencomb_mode, gencomb_median, gencomb_max, gencomb_min = (
    genres_combinations["count"].mean(),
    genres_combinations["count"].mode()[0],
    genres_combinations["count"].median(),
    genres_combinations["count"].max(),
    genres_combinations["count"].min(),
)
gen_mean, gen_mode, gen_median, gen_max, gen_min = (
    genres_count.mean(),
    genres_count.mode()[0],
    genres_count.median(),
    genres_count.max(),
    genres_count.min(),
)
most_popular_genres = genres_count[genres_count.values > gen_median]

genres_analysis = f"""
Genres Combination Analysis
================================================
Combinations Count:   {genres_combinations.count().values[1]}
Combinations Mean:    {gencomb_mean:.0f}
Combinations Mode:    {gencomb_mode}
Combinations Median:  {gencomb_median:.0f}
Combinations Max:     {gencomb_max}
Combinations Min:     {gencomb_min}
================================================
Genres Analysis
Genres Count:    {genres_count.count()}
Genres Mean:     {gen_mean:.0f}
Genres Mode:     {gen_mode}
Genres Median:   {gen_median:.0f}
Genres Max:      {gen_max}
Genres Min:      {gen_min}
"""
print(genres_analysis)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("Genres Analysis", fontsize=16)

popular_genres_combinations_plot = sns.barplot(
    data=most_popular_genres_combinations,
    x="count",
    y="genres",
    ax=axes[0, 0],
    palette="Dark2",
)
axes[0, 0].set_title("Most popular combinations of genres")
axes[0, 0].set(xlabel="Appearances", ylabel=None)
add_bar_value(popular_genres_combinations_plot)

sns.scatterplot(data=genres_cobinations_count, x="frequency", y="count", ax=axes[0, 1])
axes[0, 1].set_title("Frequency of Appearances (log-log scale)")
axes[0, 1].set_xscale("log")
axes[0, 1].set_yscale("log")
axes[0, 1].set(xlabel="Frequency", ylabel="Appearances of combinations")

popular_genres_plot = sns.barplot(
    x=most_popular_genres.values,
    y=most_popular_genres.index,
    ax=axes[1, 0],
    palette="Dark2",
)
axes[1, 0].set_title("Most popular genres")
axes[1, 0].set(xlabel="Appearances", ylabel=None)
add_bar_value(popular_genres_plot)

sns.lineplot(data=genres_count.values, ax=axes[1, 1], palette="Dark2")
axes[1, 1].set_title("Genres Distribution")
axes[1, 1].set(xlabel=None, ylabel="Frequency")

plt.show()

In [ ]:
# Categories analysis
splitted_categories = data["categories"].str.split(";")
categories_count = splitted_categories.explode().value_counts()
splitted_categories = splitted_categories[splitted_categories.str.len() > 1]
popular_categories_combination = splitted_categories.value_counts().reset_index(
    name="count"
)

most_popular_categories_combinations = popular_categories_combination.head(10)
most_popular_categories_combinations["categories"] = (
    most_popular_categories_combinations["categories"].str.join(";")
)
most_popular_categories_combinations["categories"] = (
    most_popular_categories_combinations["categories"].apply(
        lambda x: "\n".join(x.split(";"))
    )
)
categories_cobinations_count = (
    popular_categories_combination["count"].value_counts().reset_index(name="frequency")
)

catcomb_mean, catcomb_mode, catcomb_median, catcomb_max, catcomb_min = (
    popular_categories_combination["count"].mean(),
    popular_categories_combination["count"].mode()[0],
    popular_categories_combination["count"].median(),
    popular_categories_combination["count"].max(),
    popular_categories_combination["count"].min(),
)
cat_mean, cat_mode, cat_median, cat_max, cat_min = (
    categories_count.mean(),
    categories_count.mode()[0],
    categories_count.median(),
    categories_count.max(),
    categories_count.min(),
)
most_popular_categories = categories_count[categories_count.values > cat_mean]

categories_analysis = f"""
Categories Combination Analysis
================================================
Combinations Count:   {popular_categories_combination.shape[0]}
Combinations Mean:    {catcomb_mean:.0f}
Combinations Mode:    {catcomb_mode}
Combinations Median:  {catcomb_median:.0f}
Combinations Max:     {catcomb_max}
Combinations Min:     {catcomb_min}
================================================
Categories Analysis
Categories Count:    {categories_count.shape[0]}
Categories Mean:     {cat_mean:.0f}
Categories Mode:     {cat_mode}
Categories Median:   {cat_median:.0f}
Categories Max:      {cat_max}
Categories Min:      {cat_min}
"""
print(categories_analysis)

# Categories Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Categories Analysis", fontsize=16)

popular_categories_plot = sns.barplot(
    x=most_popular_categories.values,
    y=most_popular_categories.index,
    ax=axes[0],
    palette="Dark2",
)
axes[0].set_title("Most popular categories")
axes[0].set(xlabel="Appearances", ylabel=None)
add_bar_value(popular_categories_plot)

sns.lineplot(data=categories_count.values, ax=axes[1], palette="Dark2")
axes[1].set_title("Categories Distribution")
axes[1].set(xlabel=None, ylabel="Frequency")
plt.show()

# Categories Combination Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle("Categories Combination Analysis", fontsize=16)

popular_categories_combinations_plot = sns.barplot(
    data=most_popular_categories_combinations,
    x="count",
    y="categories",
    ax=axes[0],
    palette="Dark2",
)
axes[0].set_title("Most popular combinations of categories")
axes[0].set(xlabel="Appearances", ylabel=None)
add_bar_value(popular_categories_combinations_plot)
axes[0].tick_params(axis="y", labelsize=7)

sns.scatterplot(data=categories_cobinations_count, x="frequency", y="count", ax=axes[1])
axes[1].set_title("Frequency of Appearances (log-log scale)")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set(xlabel="Frequency", ylabel="Appearances of combinations")
plt.show()

In [ ]:
# Test One-Hot Encoding to analyze the correlation between categories and recommendations
one_hot_encoding = data[["release_year", "release_date", "price", "recommendations"]]
one_hot_categories = data["categories"].str.get_dummies(";")
one_hot_genres = data["genres"].str.get_dummies(";")
one_hot_encoding = pd.concat(
    [one_hot_encoding, one_hot_genres, one_hot_categories], axis=1
)
correlations = one_hot_encoding.corr()
strong_corr = (
    correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=abs, ascending=False)
)

print(strong_corr[abs(strong_corr) > 0.6])

In [ ]:
# Price analysis
n_prices = data["price"].value_counts().head(10)
most_expensive_games = data.sort_values(by="price", ascending=False).head(10)[
    ["name", "price", "recommendations", "release_year"]
]
price_per_year_month = (
    data.groupby(["release_year", data["release_date"].dt.month])["price"]
    .median()
    .reset_index(name="price")
)
n_f2p_paid_games = (
    data["price"].apply(lambda x: "F2P" if x == 0 else "Paid").value_counts()
)
print(f'\n---Price--- \n{data["price"].describe()}')
print(f"\n---Most expensive games---\n{most_expensive_games}")
print(f"""\n---Paid and Free to Play games---
    Free to Play games: {n_f2p_paid_games['F2P']}
    Paid games: {n_f2p_paid_games['Paid']}
""")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Price Analysis", fontsize=16)

sns.kdeplot(np.log1p(data["price"]), ax=axes[0])
axes[0].set_title("Price Distribution (log1p scale)")
axes[0].set(xlabel="Price", ylabel="Density")

price_barplot = sns.barplot(
    x=n_prices.index, y=n_prices.values, ax=axes[1], hue=n_prices.index, legend=False
)
axes[1].set_title("Common Price Frequency")
axes[1].set(xlabel="Price", ylabel=None)
add_bar_value(price_barplot)

f2p_paid_barplot = sns.barplot(
    x=n_f2p_paid_games.index,
    y=n_f2p_paid_games.values,
    ax=axes[2],
    hue=n_f2p_paid_games.index,
    legend=False,
)
axes[2].set_title("Free to Play vs Paid Games")
axes[2].set(xlabel="Game Type", ylabel=None)
add_bar_value(f2p_paid_barplot)
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fig.suptitle("Median Price by Year and Month", fontsize=16)
price_years = price_per_year_month["release_year"].unique()
price_palette = sns.color_palette("dark", n_colors=len(price_years))
n_price_plots = 0

for i in price_years:
    subset = price_per_year_month[price_per_year_month["release_year"] == i]
    sns.lineplot(
        data=subset,
        x="release_date",
        y="price",
        ax=axes[n_price_plots],
        color=price_palette[n_price_plots],
    )
    axes[n_price_plots].set_title(f"Monthly Median Price in {i}")
    n_price_plots += 1
plt.show()

In [ ]:
# recommendations
recommendations_count = data["recommendations"].value_counts().reset_index(name="count")
most_recommended_games = data.sort_values(by="recommendations", ascending=False).head(
    10
)[["name", "recommendations", "release_year"]]
print(f'Recommendations mode: {recommendations_count["count"].mode()[0]}')
print(f'Recommendations median: {recommendations_count["count"].median()}')
print(f'Games with 0 recommendations: {recommendations_count["count"].head(1).values[0]}')
print(f'Games with 1 recommendation: {recommendations_count[recommendations_count["count"] == 1].shape[0]}')
print(f'Games with more than 10 recommendations: {recommendations_count[recommendations_count["count"] > 10].shape[0]}')
print(f"Most recommended games:\n{most_recommended_games}")

recommendations_count_frequency = (
    recommendations_count["count"].value_counts().reset_index(name="frequency")
)
recommendations_count_frequency["frequency"] = np.log1p(
    recommendations_count_frequency["frequency"]
)
data["recommendations"] = np.log1p(data["recommendations"])

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.kdeplot(data["recommendations"], fill=True, color="green", ax=ax[0])
ax[0].set_title("Recommendations Distribution (log1p scale)")

sns.kdeplot(data=recommendations_count_frequency["frequency"], fill=True, ax=ax[1])
ax[1].set_title("Frequency of Recommendations Frequency (log1p scale)")
ax[1].set_xticks(range(0, 11))
plt.show()

In [ ]:
# developer and publisher
developer_is_publisher = data["developer"] == data["publisher"]
developer_count = data["developer"].value_counts().head(10)
publisher_count = data["publisher"].value_counts().head(10)
print(f"Number of games where developer is publisher: {developer_is_publisher.sum()}")
print(f'Number of unique developers: {data["developer"].unique().size}')
print(f'Number of unique publishers: {data["publisher"].unique().size}')
print(f"\nTop 10 developers: {developer_count}")
print(f"\nTop 10 publishers: {publisher_count}")

## Multivariate Analysis

In [ ]:
# Relation between Free/Paid Games popularity
f2p = data["price"] == 0
free_games = data[f2p]
paid_games = data[~f2p]


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Relation between Free/Paid Games and popularity (log1p scale)",
    fontsize=16,
    fontweight="bold",
    y=1.05,
)

sns.kdeplot(
    free_games["recommendations"],
    ax=axes[0],
    fill=True,
    color="green",
    label=f"Total: {len(free_games)}",
)
axes[0].set_title("Free Games")
axes[0].legend()

sns.kdeplot(
    paid_games["recommendations"],
    ax=axes[1],
    fill=True,
    color="red",
    label=f"Total: {len(paid_games)}",
)
axes[1].set_title("Paid Games")
axes[1].legend()

plt.show()

In [ ]:
# Relation between Free/Paid popular games
popular_games = data["recommendations"] > 0
free_popular_games = data[f2p & popular_games]
paid_popular_games = data[~f2p & popular_games]

print(f"Free popular games:\n {free_popular_games.describe()}")
print(f"Paid popular games:\n {paid_popular_games.describe()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Relation between Free/Paid popular games", fontsize=16, fontweight="bold", y=1.05
)

sns.kdeplot(
    free_popular_games["recommendations"],
    ax=axes[0],
    fill=True,
    color="green",
    label=f"Total: {len(free_popular_games)}",
)
axes[0].set_title("Free Popular Games")
axes[0].legend()

sns.kdeplot(
    paid_popular_games["recommendations"],
    ax=axes[1],
    fill=True,
    color="red",
    label=f"Total: {len(paid_popular_games)}",
)
axes[1].set_title("Paid Popular Games")
axes[1].legend()

plt.show()

In [ ]:
# Recommendations over time
scatterdata = data[data["recommendations"] > 0]
scatterdata["timestamp"] = pd.to_datetime(scatterdata["release_date"])
sns.scatterplot(data=scatterdata, x="timestamp", y="recommendations")
plt.title("Recommendations over time")
plt.xlabel("Release Date")
plt.ylabel("Recommendations")
plt.show()

In [ ]:
# Insights during covid period
Pandemic = (data["release_date"] >= "2020-03-11") & (
    data["release_date"] <= "2023-05-05"
)
No_Pandemic_ = ~Pandemic
print(f'Average price during the pandemic: {data[Pandemic]["price"].mean()}')
print(f'Average price outside the pandemic: {data[No_Pandemic_]["price"].mean()}')
print(f'Average recommendations during the pandemic: {data[Pandemic]["recommendations"].mean()}')
print(f'Average recommendations outside the pandemic: {data[No_Pandemic_]["recommendations"].mean()}')